# WBTC vs. BTC: 1-Minute Technical-Analysis Basis Arbitrage

Same pair as `wbtc_btc_trend_arb.ipynb` (WBTC vs. BTC on Kraken), tested with 1-minute intraday data
and technical-analysis signal constructs instead of daily rolling-regression slopes.

**A hard data constraint, stated up front and different from the OKX-based notebooks in this
series**: Kraken's public `OHLC` endpoint caps 1-minute data at its most recent **~720 candles (12
hours)** — confirmed empirically: varying the `since` parameter to anywhere from "just now" to
"3 days ago" returns the *identical* most-recent-720-candle window every time. Unlike OKX's
`history-candles` endpoint (which genuinely paginates backward), Kraken's OHLC endpoint has no real
1-minute backfill mechanism available for free. **This notebook's entire sample is whatever ~12
hours happens to be available at the moment it's run** — treat every number below as a snapshot of
one specific half-day, not a representative backtest. Re-running this notebook on a different day
will produce a completely different, equally-short sample.

**Trend lines, made tradeable without lookahead bias**: same construct as the other notebooks in
this series — geometric trendlines through swing highs/lows for visual intuition (Section 3), and a
fully-causal rolling channel breakout (Donchian-style) as the actual tradeable trend signal.

**Price deviation**: the same WBTC/BTC basis z-score construct as the daily notebook, with a much
shorter, intraday-appropriate lookback window scaled down to fit inside the available ~12-hour
sample.

> ⚠️ Research only, not investment advice. Given the ~12-hour sample, treat every result here as
> illustrative of the method, not a real finding about WBTC/BTC basis behavior.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection (1-Minute — Kraken's Full Available Window, No Pagination Possible)


In [ ]:
KRAKEN_BASE = "https://api.kraken.com/0/public/OHLC"


def fetch_kraken_ohlc(pair, interval=1):
    r = requests.get(KRAKEN_BASE, params={"pair": pair, "interval": interval}, timeout=20)
    r.raise_for_status()
    payload = r.json()
    if payload.get("error"):
        raise RuntimeError(payload["error"])
    result_key = [k for k in payload["result"] if k != "last"][0]
    rows = payload["result"][result_key]
    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close", "vwap", "volume", "count"])
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="s")
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "volume"]]


wbtc_xbt = fetch_kraken_ohlc("WBTCXBT", interval=1)
xbt_usd = fetch_kraken_ohlc("XBTUSD", interval=1)

print(f"WBTCXBT: {wbtc_xbt.index.min()} to {wbtc_xbt.index.max()}, {len(wbtc_xbt)} bars")
print(f"XBTUSD:  {xbt_usd.index.min()} to {xbt_usd.index.max()}, {len(xbt_usd)} bars")


## 2. Basis Construction


In [ ]:
df = pd.DataFrame({
    "wbtc_btc_rate": wbtc_xbt["close"],
    "btc_usd": xbt_usd["close"],
}).dropna()

df["basis_pct"] = df["wbtc_btc_rate"] - 1

print(f"Overlapping sample: {df.index.min()} to {df.index.max()}, {len(df)} minutes")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["btc_usd"], color="tab:blue")
axes[0].set_title("BTC-USD Price — 1-Minute")

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green", linewidth=0.6)
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("WBTC/BTC Basis: (WBTCXBT rate - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("With only ~12 hours of data, treat this test as illustrative — ADF has limited power on "
      "such a short sample regardless of the true underlying process.")


## 3. Trend Lines: Geometric (Visual) and Channel Breakout (Tradeable)

Windows are scaled down from the OKX-based notebooks in this series (which had ~5 days to work
with) to fit meaningfully inside this ~12-hour sample.


In [ ]:
def find_pivots(price, window=8):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=4):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).total_seconds().values / 60.0
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


swing_highs, swing_lows = find_pivots(df["btc_usd"], window=8)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df.index, df["btc_usd"], color="tab:blue", label="BTC-USD close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=20, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=20, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, df.index[-1], freq="min")
        minutes_elapsed = (x_line - x0).total_seconds() / 60.0
        y_line = intercept + slope * minutes_elapsed
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("BTC: Swing-High / Swing-Low Trend Lines (Visual Only)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
CHANNEL_WINDOW = 15  # minutes — shorter than the OKX notebooks given the compressed sample

df["channel_high"] = df["btc_usd"].rolling(CHANNEL_WINDOW).max().shift(1)
df["channel_low"] = df["btc_usd"].rolling(CHANNEL_WINDOW).min().shift(1)

df["breakout_up"] = df["btc_usd"] > df["channel_high"]
df["breakout_down"] = df["btc_usd"] < df["channel_low"]
df["in_breakout"] = df["breakout_up"] | df["breakout_down"]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df.index, df["btc_usd"], color="tab:blue", linewidth=0.8, label="BTC-USD close")
ax.plot(df.index, df["channel_high"], color="red", linewidth=0.8, linestyle="--",
        label=f"{CHANNEL_WINDOW}min rolling high")
ax.plot(df.index, df["channel_low"], color="green", linewidth=0.8, linestyle="--",
        label=f"{CHANNEL_WINDOW}min rolling low")
breakout_pts = df.index[df["in_breakout"]]
ax.scatter(breakout_pts, df.loc[breakout_pts, "btc_usd"], color="black", s=8, zorder=3, label="breakout bar")
ax.set_title(f"BTC: {CHANNEL_WINDOW}-Minute Channel Breakout Signal (Tradeable)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Breakout-up: {df['breakout_up'].sum()} bars, breakout-down: {df['breakout_down'].sum()} bars, "
      f"total breakout: {df['in_breakout'].mean()*100:.1f}% of the sample")


## 4. Does a BTC Channel Breakout Predict the WBTC Basis?


In [ ]:
def zscore(s, window=30):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

Z_WINDOW = 30  # minutes
df["basis_z"] = zscore(df["basis_pct"], window=Z_WINDOW)

breakout_basis = df.loc[df["in_breakout"], "basis_z"].dropna()
calm_basis = df.loc[~df["in_breakout"], "basis_z"].dropna()

print(f"Mean |basis_z| during channel breakouts: {breakout_basis.abs().mean():.3f} (n={len(breakout_basis)})")
print(f"Mean |basis_z| outside breakouts:         {calm_basis.abs().mean():.3f} (n={len(calm_basis)})")
print("With this few breakout minutes in a 12-hour sample, treat this comparison as a rough "
      "directional read, not a statistically powered test.")

fig, ax = plt.subplots()
ax.hist(calm_basis, bins=30, alpha=0.5, density=True, label="outside breakout")
ax.hist(breakout_basis, bins=30, alpha=0.5, density=True, label="during breakout")
ax.set_title("Basis Z-Score Distribution: Breakout vs. Calm Periods")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Mean-Reversion Arbitrage Signal (on the Basis)


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)
print(f"Pure MR: {(position_pure_mr.diff().abs() > 0).sum()} trades over {len(df)} minutes")


## 6. Combining Both Ideas: Channel-Breakout-Filtered Arbitrage


In [ ]:
position_breakout_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=df["in_breakout"])

print(f"Minutes blocked by breakout filter: {df['in_breakout'].sum()} / {len(df)} "
      f"({df['in_breakout'].mean()*100:.1f}%)")
print(f"Breakout-filtered: {(position_breakout_filtered.diff().abs() > 0).sum()} trades")


In [ ]:
MINUTES_PER_YEAR = 365 * 24 * 60


def sharpe(returns, periods_per_year=MINUTES_PER_YEAR):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "breakout_filtered_mean_reversion": position_breakout_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Breakout-Filtered WBTC/BTC Basis Arbitrage (1-Minute)")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(MINUTES_PER_YEAR),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "breakout_filtered_mean_reversion": (position_breakout_filtered.diff().abs() > 0).sum(),
    }),
})
summary


> ⚠️ **The annualized Sharpe above is close to meaningless here, more so than anywhere else in this
> series**: annualizing a **12-hour** sample via `sqrt(minutes_per_year)` multiplies by a factor of
> roughly 700x. Any small, possibly-lucky move within this half-day gets blown up into an
> astronomical "annual" number. Use `total_return` (the actual, un-annualized return over the ~12
> hours fetched) — even that is one specific half-day, not a backtest.


## 7. Market-Neutrality Check


In [ ]:
btc_ret = np.log(df["btc_usd"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_btc": [strategies["pure_mean_reversion"].corr(btc_ret)],
    "breakout_filtered_vs_btc": [strategies["breakout_filtered_mean_reversion"].corr(btc_ret)],
}, index=["correlation_with_btc_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?


In [ ]:
TAKER_FEE_PER_FILL = 0.0005
FILLS_PER_ROUND_TRIP = 4

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways

- **This notebook's sample is one specific ~12-hour window, full stop**: Kraken's public OHLC
  endpoint has no real 1-minute backfill mechanism (confirmed: `since` doesn't change what's
  returned beyond the most recent ~720 candles). Every number above describes *this run*, not
  WBTC/BTC basis behavior in general — re-running tomorrow will fetch a completely different
  12 hours.
- **If you need a real 1-minute backtest for this pair**, the practical options are: (a) poll this
  notebook's fetch cells repeatedly over time and accumulate your own local history (append each
  run's new candles to a growing on-disk dataset, similar to the caching pattern in
  `flightDataResearch`'s `data_cache/`), or (b) use a paid historical tick/minute-bar data vendor
  that retains deep history.
- **Compare directionally against the OKX-based 1-minute notebooks in this series** (which do have
  ~5 days of real data) rather than trusting this one's specific numbers in isolation.
- **Robustness**: none of the standard robustness checks (out-of-sample validation, longer-horizon
  significance) are meaningful on a single 12-hour sample — treat this notebook as a demonstration of
  method, not a result.
